# 02 — Agent: prompt → sample k programs → execute → vote → submission

Run on a split (headless: `papermill 02_agent.ipynb out.ipynb -p SPLIT public_test`, see README):
writes `<OUT>/submission.jsonl` (appended per business, so a timeout keeps finished answers) and
`<OUT>/logs/<id>.json` per question with prompts, every sample's code, stdout and answer (Rule 8).
A rerun into the same OUT resumes. Cells tagged `lib` are the agent's code; `03_evaluate` loads them.

1. Question normalization · 2. Retrieval · 3. Prompts · 4. Answers · 5. Voting · 6. LLM · 7. Agent · 8. Run

In [ ]:
SPLIT = "train"                 # "train" | "public_test" | a split folder such as "/data"
OUT = None                      # default: runs/<PROMPT>_<split name>
IDS = None                      # file with one question id per line, e.g. "../splits/val300.txt"
PROMPT = "v3"
K = 5                           # samples per question (1 means greedy decoding)
TEMPERATURE = 0.7
TAU = 0.6                       # answer only if this share of the K samples agree
MAX_REPAIRS = 1
THINKING = 0                    # thinking token budget (0 = off); Gemma 4 thinks until done otherwise
RULES_K = 0                     # rulebook rows repeated next to the question (0 turns it off, see §2)
RULE_RETRIEVER = "lexical"      # lexical | dense[:model] | hybrid
SHARD = "0/1"                   # "i/n": this process answers businesses i, i+n, i+2n, ... (one per GPU)
LLM = "vllm"                    # "vllm" or "scripted" (tests, no GPU)
SCRIPTED_REPLIES = None         # scripted: the replies every question gets as its K samples
RUN = True

In [ ]:
import time
from dataclasses import asdict, dataclass, field
from typing import Protocol

from datamate.nblib import run_lib_cells
from datamate.sandbox import ExecResult, Executor
from datamate.scoring import VALUE_TYPES, norm_answer

run_lib_cells("01_data.ipynb", globals())

## 1. Question normalization
Chat abbreviations expanded next to the question (the original stays the reference). General
language knowledge, not split statistics: 'hqua' is 'hôm qua', which v1 read as 'last month'.
Bare 'nhiu' is left alone: 'mua nhiu đơn nhất' means 'nhiều', expanding it flips the question.

In [ ]:
_ABBREV = [
    (r"dthu|d\.thu|doanh thuu", "doanh thu"),
    (r"hqua|hom qua|hôm wa|hqa", "hôm qua"),
    (r"hnay|hom nay|bữa nay|bua nay", "hôm nay"),
    (r"bnhiu|bnhieu|bnh|bn|bao nhiu|bao nhieu|bao nhiêu", "bao nhiêu"),
    (r"tháng ni|thang ni", "tháng này"),
    (r"tuần trc|tuan trc|tuần rùi|tuan rui", "tuần trước"),
    (r"trc", "trước"),
    (r"sp", "sản phẩm"),
    (r"đc|dc|đk", "được"),
    (r"ko|k|hok|hông|hong|kh0", "không"),
    (r"mik|mk|mjk", "mình"),
    (r"kua|của", "của"),
    (r"kó|ko' có", "có"),
    (r"cko|ch0", "cho"),
    (r"jùm|giùm|dùm|gium|jum", "giùm"),
    (r"nhứt|nhất", "nhất"),
    (r"ncc", "nhà cung cấp"),
    (r"nv", "nhân viên"),
    (r"sl", "số lượng"),
    (r"o đờ|ô đờ|order|bill", "đơn"),
    (r"vs", "với"),
    (r"z|zậy|dị|rứa", "vậy"),
    (r"mô", "nào"),
    (r"răng", "sao"),
]
# Each alternation grouped so both boundaries apply to every alternative; 'check', 'kém', '59-K1' stay.
_ABBREV_RX = [(re.compile(rf"(?<!\w)(?:{p})(?!\w)", re.I), r) for p, r in _ABBREV]
_MONTH_ABBREV = re.compile(r"(?<!\w)(?:th|t)\s?(?=\d{1,2}(?:\s*[/\-]\s*\d{2,4}|\b))", re.I)   # th5/2026


def expand_abbreviations(q: str) -> str:
    out = _MONTH_ABBREV.sub("tháng ", q)
    for rx, rep in _ABBREV_RX:
        out = rx.sub(rep, out)
    return re.sub(r"\s+", " ", out).strip()

## 2. Retrieval
**Value linking** (CHESS-style): names in the shop's data whose words occur in the question are
listed next to it, so the model filters on a real spelling ('Trà vải (M)', 'Natri clorid').

**Rule retrieval** (rulebook rows repeated next to the question) is off by default. On 48
hand-labelled train questions (03_evaluate §3) lexical R@5 0.47 / P@3 0.31, Qwen3-Embedding-0.6B
0.31 / 0.19, bge-m3 0.36 / 0.21, RRF hybrids at most lexical. At that precision two of three
injected rows are irrelevant, and irrelevant context distracts (Shi et al. 2023).

In [ ]:
TOKEN = re.compile(r"[a-z0-9]+")
# Filler without rule meaning (folded). Time words stay: they select the §2 period rules.
STOP = set("""la cua va co cho bao nhieu bn bnhiu nhiu may gi nao the thi nhe a ha z vay nha oi e em anh chi minh
toi ban giup xem check coi giu gium jum dc duoc da dang trong nhung cac mot""".split())


def tokens(s: str) -> list[str]:
    return [t for t in TOKEN.findall(fold(s)) if t not in STOP and len(t) > 1 and not any(c.isdigit() for c in t)]


class ValueIndex:
    """Distinct short string values of a business, by folded token set."""

    def __init__(self, biz: Business, max_distinct: int = 3000, max_len: int = 80):
        self.entries, seen = [], set()   # (value, table, column, tokens)
        for name in main_keys(biz):
            df = biz.frames[name]
            for c in df.columns:
                col = df[c].dropna()
                if col.empty or not isinstance(col.iloc[0], str):
                    continue
                vals = col.astype(str).unique()
                if len(vals) > max_distinct:
                    continue
                for v in vals:
                    if len(v) > max_len or not re.search(r"[A-Za-zÀ-ỹ]", v) or (fold(v), str(c)) in seen:
                        continue
                    seen.add((fold(v), str(c)))
                    toks = frozenset(TOKEN.findall(fold(v)))
                    if toks:
                        self.entries.append((v, name, str(c), toks))

    def match(self, question: str, k: int = 8) -> list[tuple[str, str, str]]:
        """Values all of whose words occur in the question, or names sharing >= 3 content words and
        >= 2/3 of their words with it (filler like 'bao nhiêu' must not make 'La Bảo Hoàng' a match)."""
        q_all, q_content = set(TOKEN.findall(fold(question))), set(tokens(question))
        hits = []
        for v, table, col, toks in self.entries:
            full = toks <= q_all
            inter = len(toks & q_content)
            if full or (inter >= 3 and inter / len(toks) >= 2 / 3):
                hits.append((len(toks) if full else inter, len(toks), v, table, col))
        hits.sort(key=lambda h: (-h[0], h[1]))
        out, seen = [], set()
        for _, _, v, table, col in hits:
            if (v, col) not in seen:
                seen.add((v, col))
                out.append((v, table, col))
            if len(out) == k:
                break
        return out


def value_tokens(hits) -> frozenset:
    return frozenset(t for v, _, _ in hits for t in TOKEN.findall(fold(v)))


def format_values(hits) -> str:
    return "\n".join(f"- {v!r} in tables[{t!r}][{c!r}]" for v, t, c in hits)


@dataclass(frozen=True)
class Rule:
    section: str
    name: str
    text: str


def parse_rules(rulebook: str) -> list[Rule]:
    rules, section = [], ""
    for line in rulebook.splitlines():
        if line.startswith("#"):
            section = line.lstrip("# ").strip()
            continue
        m = re.match(r"^\|\s*([^|]+?)\s*\|\s*(.+?)\s*\|\s*$", line)
        if m and not set(m[1]) <= set("-: ") and fold(m[1]) not in ("luat", "cach noi", "cach ghi", "tu", "type"):
            rules.append(Rule(section, m[1], m[2]))
    return rules


def format_rules(rules: list[Rule]) -> str:
    return "\n".join(f"- [{r.section}] {r.name}: {r.text}" for r in rules)


class LexicalRuleRetriever:
    """IDF-weighted word overlap; rule names count three times their text."""

    def __init__(self, rulebook: str):
        self.rules = parse_rules(rulebook)
        self.docs = [Counter(tokens(r.name) * 3 + tokens(r.text)) for r in self.rules]
        df = Counter(t for d in self.docs for t in d)
        self.idf = {t: math.log((len(self.docs) + 1) / (c + 0.5)) for t, c in df.items()}

    def rank(self, question: str, exclude=frozenset()) -> list[int]:
        # Rules are about concepts: a rare word inside a product name must not pull in a rule.
        q = set(tokens(expand_abbreviations(question))) - set(exclude)
        scored = [(sum(self.idf.get(t, 0) * min(d[t], 3) for t in q), i) for i, d in enumerate(self.docs)]
        return [i for s, i in sorted(scored, key=lambda x: (-x[0], x[1])) if s > 0]


QUERY_INSTRUCT = ("Instruct: Given a shop owner's question in Vietnamese, retrieve the business rule that "
                  "defines how to compute the answer\nQuery: ")


class DenseRuleRetriever:
    """Bi-encoder cosine similarity (sentence-transformers). Weights count toward the 15B budget."""

    def __init__(self, rulebook: str, model: str = "Qwen/Qwen3-Embedding-0.6B"):
        from sentence_transformers import SentenceTransformer
        self.rules, self.model = parse_rules(rulebook), model
        self.emb = SentenceTransformer(model, device=os.environ.get("DATAMATE_EMB_DEVICE", "cpu"))
        self.doc = self.emb.encode([f"{r.section} | {r.name}: {r.text}" for r in self.rules],
                                   normalize_embeddings=True, batch_size=16, show_progress_bar=False)

    def rank(self, question: str, exclude=frozenset()) -> list[int]:
        text = expand_abbreviations(question)
        text = QUERY_INSTRUCT + text if "qwen3-embedding" in self.model.lower() else text
        sims = self.emb.encode([text], normalize_embeddings=True, show_progress_bar=False)[0] @ self.doc.T
        return sorted(range(len(self.rules)), key=lambda i: -sims[i])


def rrf(rankings: list[list[int]], k: int = 60) -> list[int]:
    """Reciprocal rank fusion: combines rankings by position, so their scores need no calibration."""
    score: dict[int, float] = {}
    for ranking in rankings:
        for pos, item in enumerate(ranking):
            score[item] = score.get(item, 0.0) + 1.0 / (k + pos + 1)
    return sorted(score, key=lambda i: (-score[i], i))


class HybridRuleRetriever:
    def __init__(self, retrievers):
        self.parts, self.rules = retrievers, retrievers[0].rules

    def rank(self, question: str, exclude=frozenset()) -> list[int]:
        return rrf([r.rank(question, exclude) for r in self.parts])


def make_rule_retriever(kind: str, rulebook: str):
    if kind == "lexical":
        return LexicalRuleRetriever(rulebook)
    if kind.startswith("dense"):
        return DenseRuleRetriever(rulebook, kind.partition(":")[2] or "Qwen/Qwen3-Embedding-0.6B")
    if kind == "hybrid":
        return HybridRuleRetriever([LexicalRuleRetriever(rulebook), DenseRuleRetriever(rulebook),
                                    DenseRuleRetriever(rulebook, "BAAI/bge-m3")])
    raise ValueError(f"unknown rule retriever {kind!r}")

## 3. Prompts
Zero-shot on purpose: worked examples from train would anchor the model on train's shops, rules and
answer types, while the private test brings new industries and its own rulebook. The system message
is identical for every question of a business, so vLLM's prefix cache reuses it.

- **v2** (kept for A/B): tells the model to parse raw cells with helpers. On 30 validation questions
  it answered 0 when its own pd.to_datetime turned every date into NaT.
- **v3**: the tables arrive normalized (01_data §A3); the model uses `__date` / `__num` columns and
  `ALL …*` views; a sum over zero rows must be re-checked before answering 0; helpers renamed
  `parse_*` so they are not confused with pandas.

In [ ]:
from datamate.helpers import DOC as HELPERS_DOC_V2, DOC_V3 as HELPERS_DOC_V3

SYSTEM_V2 = """You are a careful data analyst for a small Vietnamese business. You answer the owner's question by writing Python (pandas) code over the business's own files. A wrong answer costs as much as a right answer earns, so precision matters more than answering everything.

# Environment (already loaded; do not read files)
- `tables`: dict[str, pandas.DataFrame], one per file / sheet / sqlite table, header row applied, cells RAW as in the file. Only the tables and columns listed under BUSINESS DATA exist; never invent a column.
- `calendar`: DataFrame of calendar.csv (date, weekday 1=Mon..7=Sun, lunar_day, lunar_month, lunar_year, lunar_leap, holiday).
- `AS_OF`: pandas.Timestamp, the "today" of the question, already set: never reassign it. Records dated after AS_OF have not happened yet.
- `pd`, `np`, `re`, `math`, `datetime`, `unicodedata` are imported.
{helpers}
# Reading the data
- Cells are raw: money like '55.000đ', '55k', '2,825tr', '1tr312'; quantities like '1 bình'; dates as datetime objects, Excel serials, '01/04/2025', '1/4' (year from the sheet name), '1 thg 6 2025' or ISO. One column can mix them. Parse with the helpers; each column's profile below says which helper reads it and the date range it covers.
- Order-level cells may be filled only on an order's first line: ffill_orders first, then dedupe_order_lines.
- Statuses and names have many spellings. For low-cardinality columns the profile lists ALL values: map each one explicitly (print the mapping) instead of matching a guessed spelling.
- The README of the business and the RULEBOOK fix what terms mean. When the question does not say otherwise, the rulebook default applies, including the industry section for this kind of business.

# How to answer
First write a short INTERPRETATION (at most 6 short lines, plain text):
1. What is measured, the exact period as start..end dates computed from AS_OF and the rulebook, filters, and the rulebook rules you apply.
2. Which tables and columns you use (keys and names exactly as listed).
3. What the code must check before computing (needed column present, rows in the period). Do NOT decide answerability here: you have not seen the data yet.
4. The answer type and format.
Then ONE ```python block, compact (about 60 lines at most).
- Parse every raw cell with the helpers (to_date / dates, to_number / money, to_qty). Never call pd.to_datetime, float() or int() on raw cells.
- Decide answerability IN THE CODE from evidence it prints: set unanswerable only if the shop does not record the information needed, or no table covers any day of the period (check the dates you parsed). Never substitute a different column, period or proxy for missing information. A period only partly covered by the data is computed on the covered part. Print intermediate values (rows kept after each filter, the period bounds, top candidates) so a reviewer can check the logic. If a filter keeps 0 rows, print it and reconsider the spelling or the period before concluding. End by assigning
ANSWER = {"type": <type>, "answer": <value>}
with type one of: number, percent, string, date, boolean, list, table, unanswerable.
- number: JSON number; money is integer VND. percent: percent points rounded to 2 decimals (12.4 means 12.4%); a decrease is negative.
- date: "YYYY-MM-DD", or "YYYY-MM" for a month. string: the name exactly as written in the data.
- list: ordered JSON array, also when the question asks for several numbers (keep the order asked).
- table: {"columns": [...], "rows": [[...], ...]}: first column the object name, then the measures in the order asked.
- unanswerable: ANSWER = {"type": "unanswerable", "reason": "..."}.

# RULEBOOK
{rulebook}

# BUSINESS DATA
{business}
"""

SYSTEM_V3 = """You are a careful data analyst for a small Vietnamese business. You answer the owner's question by writing Python (pandas) code over the business's own files. A wrong answer costs as much as a right answer earns, so precision matters more than answering everything.

# Environment (already loaded; do not read files)
- `tables`: dict[str, pandas.DataFrame], one per file / sheet / sqlite table with the header applied, plus `ALL …*` views that stack same-column tables (one per month or part) with a `__source` column. Read a table as `tables['<key>']` with its full key exactly as listed under BUSINESS DATA (file extension and `::sheet` included); never invent a table or a column.
- `calendar`: DataFrame of calendar.csv (date, weekday 1=Mon..7=Sun, lunar_day, lunar_month, lunar_year, lunar_leap, holiday).
- `AS_OF`: pandas.Timestamp, the "today" of the question, already set: never reassign it. Records dated after AS_OF have not happened yet.
- `pd`, `np`, `re`, `math`, `datetime`, `unicodedata` are imported.
{helpers}
# Reading the data
- Raw cells keep their file spelling ('55.000đ', '1tr312', '1/4', '1 thg 6 2025', Excel serials...). Parsing is already done: next to a raw date column there is `<col>__date` (datetime64 at midnight; `<col>__dt` also keeps the time) and next to a raw money or number column `<col>__num` (float). Filter, group and sum on these parsed columns, never on raw cells. Each column's profile below names its parsed column and the date range it covers.
- Where a table's profile says so, order-level cells (order id, date, customer, status...) are already filled down onto every line of an order or record. Exact duplicate lines of the same order (rulebook §8) are already removed: do not dedupe again.
- Statuses and names have many spellings. For low-cardinality columns the profile lists ALL values: map each one explicitly (print the mapping) instead of matching a guessed spelling.
- The README of the business and the RULEBOOK fix what terms mean. When the question does not say otherwise, the rulebook default applies, including the industry section for this kind of business.

# How to answer
First write a short INTERPRETATION (at most 6 short lines, plain text):
1. What is measured, the exact period as start..end dates computed from AS_OF and the rulebook, filters, and the rulebook rules you apply.
2. Which tables and columns you use (keys and names exactly as listed).
3. What the code must check before computing (needed column present, rows in the period). Do NOT decide answerability here: you have not seen the data yet.
4. The answer type and format.
Then ONE ```python block, compact (about 60 lines at most).
- Use the parsed `__date` / `__dt` / `__num` columns. Only for a raw cell that has no parsed column, use parse_date / parse_number / parse_qty. Never call pd.to_datetime, float() or int() on raw cells.
- A column that both tables of a merge have (other than the key) comes out as `<col>_x` / `<col>_y`: take only the columns you need from the other table before merging.
- A sum or count over zero rows is not an answer by itself: print how many rows each filter keeps, and if the period has data but nothing matches, re-check the spellings and the period before answering 0.
- Decide answerability IN THE CODE from evidence it prints: set unanswerable only if the shop does not record the information needed, or no table covers any day of the period (check the dates you parsed). Never substitute a different column, period or proxy for missing information. A period only partly covered by the data is computed on the covered part. Print intermediate values (rows kept after each filter, the period bounds, top candidates) so a reviewer can check the logic. If a filter keeps 0 rows, print it and reconsider the spelling or the period before concluding. End by assigning
ANSWER = {"type": <type>, "answer": <value>}
with type one of: number, percent, string, date, boolean, list, table, unanswerable.
- number: JSON number; money is integer VND. percent: percent points rounded to 2 decimals (12.4 means 12.4%); a decrease is negative.
- date: "YYYY-MM-DD", or "YYYY-MM" for a month. string: the name exactly as written in the data.
- list: ordered JSON array, also when the question asks for several numbers (keep the order asked).
- table: {"columns": [...], "rows": [[...], ...]}: first column the object name, then the measures in the order asked.
- unanswerable: ANSWER = {"type": "unanswerable", "reason": "..."}.

# RULEBOOK
{rulebook}

# BUSINESS DATA
{business}
"""

USER = "Hôm nay (as_of_date): {as_of}\nCâu hỏi: {question}"
USER_EXPANDED = "\nCâu hỏi, viết tắt đã viết đầy đủ (chỉ để dễ đọc; câu gốc ở trên là chuẩn): {expanded}"
USER_RULES = "\n\nRulebook rows that share words with the question (the full rulebook above still applies):\n{rules}"
USER_VALUES = "\n\nValues in the data that match words of the question:\n{values}"
REPAIR = """Your code failed or produced an invalid ANSWER.
{problem}
stdout (tail):
{stdout}
Fix it and reply with the full corrected ```python block."""


@dataclass(frozen=True)
class PromptSpec:
    name: str
    system: str
    helpers_doc: str

    def build_system(self, rulebook: str, business_desc: str) -> str:
        return (self.system.replace("{helpers}", self.helpers_doc).replace("{rulebook}", rulebook.strip())
                .replace("{business}", business_desc))

    def build_user(self, q: dict, rules: str = "", values: str = "") -> str:
        u = USER.format(as_of=q["as_of_date"], question=q["question"])
        exp = expand_abbreviations(q["question"])
        u += USER_EXPANDED.format(expanded=exp) if exp != q["question"] else ""
        u += USER_RULES.format(rules=rules) if rules else ""
        return u + (USER_VALUES.format(values=values) if values else "")


PROMPTS = {"v2": PromptSpec("v2", SYSTEM_V2, HELPERS_DOC_V2), "v3": PromptSpec("v3", SYSTEM_V3, HELPERS_DOC_V3)}

## 4. Answers
Extract the code, validate ANSWER against the one format each type accepts, and apply the
rulebook §9 type guard (a percentage question takes type percent; the model often types it number).

In [ ]:
_CODE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)
_ISO_DATE = re.compile(r"\d{4}-\d{2}(-\d{2})?")
# Rulebook §3: an order without a customer (walk-in) is not a customer and is never ranked. Spellings seen
# for it in the data of both splits; bare "le" only outside customer-group questions ("nhóm lẻ" is a group).
_WALK_IN = {"khach le", "kl", "khach vang lai", "vang lai"}
_CUSTOMER_Q = re.compile(r"\bkhach\b|\bai\b")
_PCT_CUE = re.compile(r"(?<![\d,.])%|phan tram|ti le|ty le|ti trong|ty trong|chiem")


def extract_code(text: str) -> str | None:
    """The longest fenced python block of the final answer: the text after the thought (Qwen's </think>,
    Gemma 4's <channel|>)."""
    text = re.split(r"</think>|<channel\|>", text)[-1]
    blocks = _CODE.findall(text)
    return max(blocks, key=len) if blocks else (text if "ANSWER" in text else None)


def _num(v):
    if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v):
        raise ValueError(f"not a number: {v!r}")
    return int(round(v)) if abs(v - round(v)) < 1e-6 else round(float(v), 2)


def _cell(x):
    return x.strip() if isinstance(x, str) else _num(x)


def normalize_answer(ans) -> dict:
    """Validate the type and coerce the value to the format the scorer accepts. Raises ValueError."""
    if not isinstance(ans, dict) or "type" not in ans:
        raise ValueError("ANSWER must be a dict with a 'type' key")
    t = ans["type"]
    if t == "unanswerable":
        return {"type": t, "reason": str(ans.get("reason", ""))[:300]}
    if t == "abstain":
        return {"type": t}
    if t not in VALUE_TYPES:
        raise ValueError(f"unknown type {t!r}")
    if ans.get("answer") is None:
        raise ValueError("missing 'answer'")
    v = ans["answer"]
    if t == "number":
        v = _num(v)
    elif t == "percent":
        v = round(float(_num(v)), 2)
    elif t == "string":
        if not isinstance(v, str):
            raise ValueError("string answer must be a str")
        v = v.strip()
    elif t == "date":
        if not (isinstance(v, str) and re.fullmatch(r"\d{4}-\d{2}(-\d{2})?", v)):
            raise ValueError(f"date must be YYYY-MM-DD or YYYY-MM, got {v!r}")
    elif t == "boolean":
        if not isinstance(v, bool):
            raise ValueError("boolean answer must be true/false")
    elif t == "list":
        if not isinstance(v, list) or not v:
            raise ValueError("list answer must be a non-empty list")
        v = [_cell(x) for x in v]
    elif t == "table":
        if not (isinstance(v, dict) and isinstance(v.get("columns"), list) and isinstance(v.get("rows"), list)):
            raise ValueError("table answer must be {'columns': [...], 'rows': [[...]]}")
        if any(not isinstance(r, list) or len(r) != len(v["columns"]) for r in v["rows"]):
            raise ValueError("every table row must have one cell per column")
        v = {"columns": [str(c) for c in v["columns"]], "rows": [[_cell(x) for x in r] for r in v["rows"]]}
    return {"type": t, "answer": v}


def _names(ans: dict) -> list[str]:
    a, t = ans.get("answer"), ans.get("type")
    if t == "string":
        return [a]
    if t == "list":
        return [x for x in a if isinstance(x, str)]
    if t == "table":
        return [r[0] for r in a["rows"] if r and isinstance(r[0], str)]
    return []


def names_walk_in(question: str, ans: dict) -> bool:
    """True when a customer question is answered with the walk-in customer (rulebook §3: not a customer)."""
    q = fold(question)
    if not _CUSTOMER_Q.search(q):
        return False
    walk_in = _WALK_IN | ({"le"} if "nhom" not in q else set())
    return any(fold(n) in walk_in for n in _names(ans))


def postprocess(question: str, ans: dict | None) -> dict | None:
    """Deterministic format fixes; None drops the sample (it then counts as a failed sample in the vote)."""
    if ans is not None and names_walk_in(question, ans):
        return None
    if ans is not None and ans.get("type") == "number" and _PCT_CUE.search(fold(question)):
        return {"type": "percent", "answer": round(float(ans["answer"]), 2)}
    if ans is not None and ans.get("type") == "string" and _ISO_DATE.fullmatch(ans["answer"]):
        return {"type": "date", "answer": ans["answer"]}   # a day or month written as a name is still a date
    return ans

## 5. Voting and abstention
Samples are clustered by the value the scorer compares (not by code text); the largest cluster
wins and its share of the K samples is the confidence. Under PA answering pays only when
P(correct) > 0.5, so TAU is tuned on validation (03_evaluate), never on the public test.
Consistent errors survive voting (semantic-entropy caveat), which is why §A3 normalization matters.

In [ ]:
def _key_cell(v):
    return round(float(v), 2) if isinstance(v, (int, float)) and not isinstance(v, bool) else norm_answer(v)


def answer_key(ans: dict) -> tuple:
    t = ans["type"]
    if t in ("unanswerable", "abstain"):
        return (t,)
    v = ans["answer"]
    if t in ("number", "percent"):
        return (t, round(float(v), 2))
    if t == "string":
        return (t, norm_answer(v))
    if t == "list":
        return (t, tuple(_key_cell(x) for x in v))
    if t == "table":
        return (t, len(v["columns"]), frozenset(tuple(_key_cell(x) for x in r) for r in v["rows"]))
    return (t, v)


def vote(answers: list, k: int) -> dict:
    """answers: one normalized answer (or None for a failed sample) per sample."""
    groups: dict[tuple, list[dict]] = {}
    for a in answers:
        if a is not None and a["type"] != "abstain":
            groups.setdefault(answer_key(a), []).append(a)
    if not groups:
        return {"answer": None, "support": 0.0, "clusters": []}
    ranked = sorted(groups.values(), key=len, reverse=True)
    best = ranked[0]
    rep = Counter(repr(a) for a in best).most_common(1)[0][0]
    return {"answer": next(a for a in best if repr(a) == rep), "support": len(best) / k,
            "clusters": [{"answer": g[0], "size": len(g)} for g in ranked]}


def decide(voted: dict, tau: float) -> dict:
    return voted["answer"] if voted["answer"] is not None and voted["support"] >= tau else {"type": "abstain"}

## 6. LLM
The agent depends on `ChatModel` only. `VLLMChatModel` reads every engine knob from environment
variables so the Docker image and a dev box run the same code; `ScriptedChat` replays fixed replies
for tests. The seed is fixed: the organizers' re-run must reproduce the upload within 0.03 PA.

In [ ]:
class ChatModel(Protocol):
    def chat(self, conversations: list[list[dict]], n: int = 1, temperature: float = 0.0,
             thinking: int = 0) -> list[list[str]]: ...


class VLLMChatModel:
    def __init__(self, model: str | None = None, seed: int = 0):
        from vllm import LLM as _LLM, SamplingParams
        model = model or os.environ.get("DATAMATE_MODEL", "google/gemma-4-12B-it")
        mm = {"image": 0, "video": 0, **({"audio": 0} if "gemma-4" in model.lower() else {})}
        self.llm = _LLM(
            model=model,
            quantization=os.environ.get("DATAMATE_QUANT", "fp8") or None,
            # Longest prompt ~15k tokens + thinking budget + reply. fp8 KV fits it beside fp8 weights on 24 GB.
            max_model_len=int(os.environ.get("DATAMATE_MAX_LEN", 24576)),
            kv_cache_dtype=os.environ.get("DATAMATE_KV_DTYPE", "fp8"),
            gpu_memory_utilization=float(os.environ.get("DATAMATE_GPU_UTIL", 0.6)),
            tensor_parallel_size=int(os.environ.get("DATAMATE_TP", 1)),
            enable_prefix_caching=True,
            # gpu_memory_utilization does not cap the profiling peak (~262k-vocab logits); fewer seqs keeps it down.
            max_num_seqs=int(os.environ.get("DATAMATE_MAX_SEQS", 16)),
            limit_mm_per_prompt=mm,
            # The parser lets vLLM end the thought when the budget runs out (thinking_token_budget).
            reasoning_parser=os.environ.get("DATAMATE_REASONING_PARSER", "gemma4") or None,
            seed=seed,
        )
        self._sampling, self.seed = SamplingParams, seed

    def chat(self, conversations, n=1, temperature=0.0, thinking=0):
        """thinking: token budget for the thought (0 = off). Special tokens stay in the text so extract_code
        can cut the thought off at its closing tag."""
        budget = 8192 if thinking is True else int(thinking)
        params = self._sampling(n=n, temperature=temperature, top_p=0.95 if temperature > 0 else 1.0,
                                top_k=64 if temperature > 0 else -1,
                                max_tokens=int(os.environ.get("DATAMATE_MAX_TOKENS", 4096)) + budget,
                                thinking_token_budget=budget or None, skip_special_tokens=False, seed=self.seed)
        outs = self.llm.chat(conversations, params, use_tqdm=False,
                             chat_template_kwargs={"enable_thinking": budget > 0})
        # Per output: generated tokens and why it stopped ('length' = cut off at max_tokens), for the logs.
        self.last_meta = [[(len(getattr(c, "token_ids", None) or []), getattr(c, "finish_reason", None))
                           for c in o.outputs] for o in outs]
        return [[c.text for c in o.outputs] for o in outs]


class ScriptedChat:
    """Replays scripted replies. Each call takes the next round (the last one repeats); a round is a list
    with one entry per conversation (cycled), each a reply or a list of replies (the n samples)."""

    def __init__(self, *rounds):
        self.rounds, self.calls = list(rounds), []

    def chat(self, conversations, n=1, temperature=0.0, thinking=0):
        self.calls.append({"n": n, "temperature": temperature, "conversations": conversations})
        replies = self.rounds.pop(0) if len(self.rounds) > 1 else self.rounds[0]
        per_conv = [r if isinstance(r, list) else [r] for r in replies]
        return [(per_conv[i % len(per_conv)] * n)[:n] for i in range(len(conversations))]

## 7. Agent
Per business: prompts → K samples → parallel sandboxed execution → one repair turn for every sample that
failed (error, no code, invalid ANSWER), whose fix takes that sample's place → vote. Measured on the public
run: about 10% of samples crash, mostly on a wrong column or table name, which the error message names.

In [ ]:
import hashlib


@dataclass
class AgentConfig:
    prompt: str = "v3"
    k: int = 5
    temperature: float = 0.7
    tau: float = 0.6
    max_repairs: int = 1
    thinking: int = 0                # thought token budget, 0 = off
    rules_k: int = 0
    rule_retriever: str = "lexical"


@dataclass
class BusinessRun:
    rows: list[dict] = field(default_factory=list)   # submission lines
    logs: list[dict] = field(default_factory=list)   # per-question logs (Rule 8)
    llm_seconds: float = 0.0
    total_seconds: float = 0.0
    gen_tokens: int = 0       # tokens generated, when the model reports them
    truncated: int = 0        # outputs cut off at max_tokens


def _settle(r: ExecResult, log: dict) -> dict | None:
    log.update(stdout=r.stdout, error=r.error, exec_seconds=round(r.elapsed, 3))
    if not r.ok:
        return None
    try:
        log["answer"] = normalize_answer(r.answer)   # kept per sample so votes can be recomputed offline
        return log["answer"]
    except ValueError as e:
        log["error"] = f"Invalid ANSWER {r.answer!r}: {e}"
        return None


def execute_replies(ex: Executor, items: list[tuple[str, str]]) -> list[tuple[dict | None, dict]]:
    """items: (reply, as_of). Runs every extracted program in parallel; results in input order."""
    attempts = []
    for reply, as_of in items:
        code = extract_code(reply)
        log = {"reply": reply, "code": code}
        if code is None:
            log["error"] = "No python code block in the reply."
        attempts.append((code, as_of, log))
    results = iter(ex.run_many([(code, as_of) for code, as_of, _ in attempts if code is not None]))
    return [(_settle(next(results), log) if code is not None else None, log) for code, _, log in attempts]


class Agent:
    def __init__(self, llm: ChatModel, cfg: AgentConfig, rulebook: str, calendar_csv: Path):
        self.llm, self.cfg, self.rulebook, self.calendar_csv = llm, cfg, rulebook, calendar_csv
        self.spec = PROMPTS[cfg.prompt]
        self.rules = make_rule_retriever(cfg.rule_retriever, rulebook) if cfg.rules_k else None

    def _user_turn(self, q: dict, values: ValueIndex) -> str:
        hits = values.match(q["question"])
        rules = ([self.rules.rules[i] for i in self.rules.rank(q["question"], value_tokens(hits))[:self.cfg.rules_k]]
                 if self.rules else [])
        return self.spec.build_user(q, rules=format_rules(rules), values=format_values(hits))

    def _chat(self, convs, n, temperature, run: BusinessRun):
        t = time.time()
        out = self.llm.chat(convs, n=n, temperature=temperature, thinking=self.cfg.thinking)
        run.llm_seconds += time.time() - t
        meta = getattr(self.llm, "last_meta", None)
        self._meta = [m for per in meta for m in per] if meta else None   # flat, in reply order
        if self._meta:
            run.gen_tokens += sum(tokens for tokens, _ in self._meta)
            run.truncated += sum(finish == "length" for _, finish in self._meta)
        return out

    def answer_business(self, biz: Business, tables_pickle: Path, questions: list[dict]) -> BusinessRun:
        cfg, run, t0 = self.cfg, BusinessRun(), time.time()
        system = self.spec.build_system(self.rulebook, describe_business(biz))
        values = ValueIndex(biz)
        users = [self._user_turn(q, values) for q in questions]
        convs = [[{"role": "system", "content": system}, {"role": "user", "content": u}] for u in users]
        system_sha = hashlib.sha256(system.encode("utf-8")).hexdigest()[:16]   # same prompt => same hash
        logs = [{"id": q["id"], "business_id": biz.biz_id, "question": q["question"], "as_of_date": q["as_of_date"],
                 "system_sha": system_sha, "user_prompt": u, "samples": []} for q, u in zip(questions, users)]
        answers = [[] for _ in questions]
        replies = self._chat(convs, cfg.k, cfg.temperature if cfg.k > 1 else 0.0, run)
        with Executor(tables_pickle, self.calendar_csv) as ex:
            owners = [i for i, texts in enumerate(replies) for _ in texts]
            items = [(text, questions[i]["as_of_date"]) for i, texts in enumerate(replies) for text in texts]
            meta = self._meta
            for f, (i, (ans, slog)) in enumerate(zip(owners, execute_replies(ex, items))):
                if meta:
                    slog.update(gen_tokens=meta[f][0], finish=meta[f][1])
                answers[i].append(postprocess(questions[i]["question"], ans))
                logs[i]["samples"].append(slog)
            self._repair(ex, convs, questions, answers, logs, run)
        run.total_seconds = time.time() - t0
        for q, ans, lg in zip(questions, answers, logs):
            voted = vote(ans, cfg.k)
            row = {"id": q["id"], **decide(voted, cfg.tau)}
            lg.update(final=row, support=voted["support"], clusters=voted["clusters"],
                      elapsed_seconds=round(run.total_seconds / len(questions), 3))
            run.rows.append(row)
            run.logs.append(lg)
        return run

    def _repair(self, ex, convs, questions, answers, logs, run):
        """Each failed sample (error, no code block, invalid ANSWER) gets its reply and error back once; the
        fixed program replaces that sample in the vote. A sample dropped by postprocess has no error and is
        not retried. `repair_of` in the log names the sample a repair replaces."""
        latest = {(i, j): s for i, lg in enumerate(logs) for j, s in enumerate(lg["samples"])}
        for _ in range(self.cfg.max_repairs):
            todo = [(i, j) for i, a in enumerate(answers) for j, x in enumerate(a)
                    if x is None and latest[(i, j)].get("error")]
            if not todo:
                return
            rconvs = []
            for i, j in todo:
                s = latest[(i, j)]
                rconvs.append(convs[i] + [
                    {"role": "assistant", "content": s["reply"]},
                    {"role": "user", "content": REPAIR.format(problem=s.get("error"),
                                                              stdout=(s.get("stdout") or "")[-1500:])}])
            fixed = self._chat(rconvs, 1, 0.0, run)
            items = [(texts[0], questions[i]["as_of_date"]) for (i, _), texts in zip(todo, fixed)]
            meta = self._meta
            for f, ((i, j), (ans, slog)) in enumerate(zip(todo, execute_replies(ex, items))):
                slog.update(repair=True, repair_of=j)
                if meta:
                    slog.update(gen_tokens=meta[f][0], finish=meta[f][1])
                logs[i]["samples"].append(slog)
                latest[(i, j)] = slog
                answers[i][j] = postprocess(questions[i]["question"], ans) if ans is not None else None

## 8. Run over a split
Shards split the businesses across processes (one per GPU); each writes `OUT/shards/<i>/`, and
`merge_shards` assembles `OUT/submission.jsonl` + `OUT/logs/` once every shard is done.

In [ ]:
def default_out(split, prompt: str) -> Path:
    return REPO / "runs" / f"{prompt}_{split_root(split).name}"


def assign_shops(counts: dict[str, int], n: int) -> list[list[str]]:
    """Shops to n workers: largest first onto the least loaded, by question count. Deterministic and computed
    from every question of the run (not only the unanswered ones), so a resumed shard keeps its shops."""
    load, out = [0] * n, [[] for _ in range(n)]
    for b in sorted(counts, key=lambda b: (-counts[b], b)):
        w = min(range(n), key=lambda w: (load[w], w))
        out[w].append(b)
        load[w] += counts[b]
    return [sorted(x) for x in out]


def _write_json(path: Path, obj) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")
    tmp.replace(path)


def _append_lines(path: Path, lines: list[str]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write("".join(line + "\n" for line in lines))
        f.flush()
        os.fsync(f.fileno())


def load_done(sub: Path, logs_dir: Path) -> set[str]:
    """Ids already answered. A torn last line (crash mid-write), a repeated id, or a row whose log is missing
    is dropped and the file rewritten without it, so a resumed run redoes exactly those questions."""
    if not sub.exists():
        return set()
    text = sub.read_text(encoding="utf-8")
    keep, done = [], set()
    for line in text.splitlines():
        try:
            qid = json.loads(line)["id"]
        except (json.JSONDecodeError, KeyError, TypeError):
            continue
        if qid not in done and (logs_dir / f"{qid}.json").exists():
            keep.append(line)
            done.add(qid)
    clean = "".join(line + "\n" for line in keep)
    if clean != text:
        tmp = sub.with_name(sub.name + ".tmp")
        tmp.write_text(clean, encoding="utf-8")
        tmp.replace(sub)
    return done


def run_manifest(cfg: AgentConfig, shard: str, root: Path, n_questions: int) -> dict:
    """What a rerun needs to reproduce this one: code version, settings, model, libraries, environment."""
    import platform
    import subprocess
    from importlib import metadata

    def git(*args):
        try:
            return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True, timeout=20).stdout
        except (OSError, subprocess.SubprocessError):
            return ""

    def version(pkg):
        try:
            return metadata.version(pkg)
        except metadata.PackageNotFoundError:
            return None

    diff = git("diff", "HEAD")
    return {"time": dt.datetime.now().isoformat(timespec="seconds"), "host": platform.node(), "pid": os.getpid(),
            "shard": shard, "split": str(root), "questions": n_questions, "config": asdict(cfg),
            "git_commit": git("rev-parse", "HEAD").strip(), "git_dirty": bool(diff),
            "git_diff_sha": hashlib.sha256(diff.encode()).hexdigest()[:16] if diff else None,
            "cache_version": CACHE_VERSION, "model": os.environ.get("DATAMATE_MODEL", "google/gemma-4-12B-it"),
            "env": {k: v for k, v in sorted(os.environ.items())
                    if k.startswith(("DATAMATE_", "VLLM_", "HF_")) or k == "CUDA_VISIBLE_DEVICES"},
            "versions": {p: version(p) for p in ("vllm", "torch", "transformers", "pandas", "numpy", "openpyxl")}}


def _prepare(root: Path, biz_id: str):
    biz = load_business_cached(root, biz_id)
    return biz, sandbox_tables(root, biz)


def run_split(split, out: Path, cfg: AgentConfig, ids: set[str] | None = None, llm: ChatModel | None = None,
              shard: str = "0/1") -> Path:
    """Answer one shard of a split, shop by shop. Per shop it writes the question logs first, then the
    submission rows (fsync), then one progress line; a rerun into the same folder resumes where it stopped.
    runs.jsonl records every start / end with the code version and settings."""
    from concurrent.futures import ThreadPoolExecutor
    i, n = map(int, shard.split("/"))
    root = split_root(split)
    work = out / "shards" / str(i) if n > 1 else out
    (work / "logs").mkdir(parents=True, exist_ok=True)
    (out / "config.json").write_text(json.dumps(asdict(cfg), indent=1))
    questions = [q for q in read_jsonl(root / "questions.jsonl") if ids is None or q["id"] in ids]
    sub = work / "submission.jsonl"
    done = load_done(sub, work / "logs")
    by_biz: dict[str, list[dict]] = {}
    for q in questions:
        if q["id"] not in done:
            by_biz.setdefault(q["business_id"], []).append(q)
    owned = assign_shops(Counter(q["business_id"] for q in questions), n)[i]
    mine = [b for b in owned if b in by_biz]
    manifest = run_manifest(cfg, shard, root, len(questions))
    _append_lines(work / "runs.jsonl", [to_json_line({"event": "start", "already_done": len(done),
                                                      "shops_todo": mine, **manifest})])
    t_all = time.time()
    if mine:
        # The model loads before the prefetch thread starts: vLLM may fork, and a fork under a running
        # thread can deadlock. After that, the next shop's tables load while the GPU works on this one.
        agent = Agent(llm or VLLMChatModel(), cfg, (root / "rulebook.md").read_text(encoding="utf-8"),
                      root / "calendar.csv")
        with ThreadPoolExecutor(max_workers=1) as pool:
            nxt = pool.submit(_prepare, root, mine[0])
            for bi, biz_id in enumerate(mine):
                biz, tables = nxt.result()
                if bi + 1 < len(mine):
                    nxt = pool.submit(_prepare, root, mine[bi + 1])
                br = agent.answer_business(biz, tables, by_biz[biz_id])
                for row, lg in zip(br.rows, br.logs):
                    _write_json(work / "logs" / f"{row['id']}.json", lg)
                _append_lines(sub, [to_json_line(row) for row in br.rows])
                answered = sum(r["type"] != "abstain" for r in br.rows)
                samples = [s for lg in br.logs for s in lg["samples"]]
                first = [s for s in samples if not s.get("repair")]
                repairs = [s for s in samples if s.get("repair")]
                _append_lines(work / "progress.jsonl", [to_json_line({
                    "time": dt.datetime.now().isoformat(timespec="seconds"), "shop": biz_id,
                    "questions": len(br.rows), "answered": answered, "samples": len(first),
                    "failed_samples": sum(bool(s.get("error")) for s in first), "repairs": len(repairs),
                    "repairs_fixed": sum(s.get("answer") is not None for s in repairs),
                    "gen_tokens": br.gen_tokens, "truncated": br.truncated,
                    "llm_seconds": round(br.llm_seconds, 1), "total_seconds": round(br.total_seconds, 1)})])
                print(f"[shard {i}/{n}] [{bi + 1}/{len(mine)}] {biz_id}: {len(br.rows)} q, {answered} answered, "
                      f"llm {br.llm_seconds:.0f}s, total {br.total_seconds:.0f}s | elapsed {time.time() - t_all:.0f}s",
                      flush=True)
    total = len(load_done(sub, work / "logs"))
    _append_lines(work / "runs.jsonl", [to_json_line({"event": "end", "time": dt.datetime.now().isoformat(
        timespec="seconds"), "pid": os.getpid(), "elapsed_seconds": round(time.time() - t_all, 1),
        "answered_ids": total, "shard_questions": sum(c for b, c in Counter(
            q["business_id"] for q in questions).items() if b in owned)})])
    if n > 1:
        merge_shards(out, n)
    return out


def merge_shards(out: Path, n: int) -> bool:
    """Write OUT/submission.jsonl and OUT/logs from every shard once all n have finished."""
    shards = [out / "shards" / str(i) / "submission.jsonl" for i in range(n)]
    if not all(p.exists() for p in shards):
        return False
    tmp = out / f"submission.{os.getpid()}.tmp"   # two shards finishing together must not share a temp file
    with tmp.open("w", encoding="utf-8") as f:
        for p in shards:
            f.write(p.read_text(encoding="utf-8"))
    (out / "logs").mkdir(exist_ok=True)
    for i in range(n):
        for lg in (out / "shards" / str(i) / "logs").glob("*.json"):
            (out / "logs" / lg.name).write_bytes(lg.read_bytes())
    tmp.replace(out / "submission.jsonl")
    return True

## Run

In [ ]:
if RUN:
    cfg = AgentConfig(PROMPT, K, TEMPERATURE, TAU, MAX_REPAIRS, THINKING, RULES_K, RULE_RETRIEVER)
    llm = ScriptedChat([SCRIPTED_REPLIES]) if LLM == "scripted" else None
    ids = set(Path(IDS).read_text().split()) if IDS else None
    out = Path(OUT) if OUT else default_out(SPLIT, PROMPT)
    run_split(SPLIT, out, cfg, ids, llm, SHARD)
    sub = out / "submission.jsonl"
    if sub.exists():
        rows = read_jsonl(sub)
        print(len(rows), "answers written to", sub, "|", dict(Counter(r["type"] for r in rows)))